In [1]:
import sys, json, time
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import os
import getpass

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")

In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="ragas")

from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.dataset_schema import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq

ragas_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
ragas_embeddings = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))

wrapped_llm = LangchainLLMWrapper(ragas_llm)
metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
for m in metrics:
    if hasattr(m, "llm"):
        m.llm = wrapped_llm
    if hasattr(m, "embeddings"):
        m.embeddings = ragas_embeddings

"""Groq's API caps n=1 per request; answer_relevancy defaults to generating 
   multiple (strictness=3) candidate questions per call, which requests n=3
   and gets rejected. Force it down to 1. Setting strictness=1 means 
   answer_relevancy averages over a single generated question instead of three,
   which makes that specific metric slightly noisier per-row 
   (less averaging = more variance). Worth fixing with OpenAI API."""
   
if hasattr(answer_relevancy, "strictness"):
    answer_relevancy.strictness = 1
    print("answer_relevancy.strictness set to 1 (Groq caps n at 1)")

/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_70713/3006640463.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_70713/3006640463.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_70713/3006640463.py:

answer_relevancy.strictness set to 1 (Groq caps n at 1)


/tmp/ipykernel_70713/3006640463.py:12: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))
/tmp/ipykernel_70713/3006640463.py:14: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  wrapped_llm = LangchainLLMWrapper(ragas_llm)


In [3]:
import asyncio

async def score_row_async(row: dict, metric) -> float:
    """Plain coroutine — awaited directly via Jupyter's native top-level
    await, never wrapped in asyncio.run(). Jupyter's kernel already runs
    inside its own event loop (needed to support await in cells at all),
    so asyncio.run() always fails immediately with 'cannot be called
    from a running event loop' — before the coroutine body even starts,
    meaning no API calls are made and no quota is spent on that failure."""
    sample = SingleTurnSample(
        user_input=row["question"], response=row["answer"],
        retrieved_contexts=row["contexts"], reference=row.get("ground_truth", ""),
    )
    return await metric.single_turn_ascore(sample)


async def call_with_backoff_async(coro_fn, max_retries: int = 4, pre_delay: float = 2.0,
                                   retryable_markers=("500", "INTERNAL", "429", "RESOURCE_EXHAUSTED")):
    """Async counterpart to rag_lab.utils.call_with_backoff. coro_fn must
    be a zero-arg callable returning a FRESH coroutine each call (e.g. a
    lambda) — a coroutine object can only be awaited once, so a naive
    retry reusing the same coroutine object would fail on attempt 2."""
    await asyncio.sleep(pre_delay)
    for attempt in range(max_retries):
        try:
            return await coro_fn()
        except Exception as e:
            from rag_lab.utils import is_daily_quota_error
            if is_daily_quota_error(e):
                raise
            msg = str(e)
            if not any(marker in msg for marker in retryable_markers):
                raise
            if attempt == max_retries - 1:
                raise
            wait = (2 ** attempt) * 2
            print(f"  Transient error, retrying in {wait:.1f}s (attempt {attempt + 1}/{max_retries}): {msg[:100]}")
            await asyncio.sleep(wait)

In [4]:
# Load Generation Results
def load_all_results(results_path):
    loaded, skipped = [], 0
    with open(results_path) as f:
        for line in f:
            record = json.loads(line)
            if "error" in record:
                skipped += 1
                continue
            loaded.append(record)
    print(f"{len(loaded)} successful results, {skipped} failed attempts skipped")
    return loaded

results = load_all_results(project_root / "eval" / "eval_results.jsonl")

218 successful results, 160 failed attempts skipped


In [5]:
from rag_lab.utils import is_daily_quota_error, load_completed_keys, append_jsonl

SCORE_ROWS_PATH = project_root / "eval" / "ragas_row_scores.jsonl"
METRIC_NAMES = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

async def run_scoring_loop_async(all_results, metrics, scores_path, pacing_delay: float = 1.5):
    # require_fields means the 155 PARTIAL rows from the broken asyncio.run()
    # attempt are correctly excluded here and will be retried below —
    # no manual cleanup of eval/ragas_row_scores.jsonl needed
    already_scored = load_completed_keys(scores_path, require_fields=METRIC_NAMES)
    print(f"{len(already_scored)} rows fully scored so far\n")

    for row in all_results:
        key = (row["strategy"], row["id"])
        if key in already_scored:
            continue

        row_scores = {"strategy": row["strategy"], "id": row["id"], "question_type": row["question_type"]}
        quota_hit = False
        for metric in metrics:
            try:
                row_scores[metric.name] = await call_with_backoff_async(
                    lambda m=metric: score_row_async(row, m)
                )
            except Exception as e:
                if is_daily_quota_error(e):
                    print(f"\nDAILY QUOTA EXHAUSTED at {row['strategy']}/{row['id']}/{metric.name}. Resume later.\n")
                    quota_hit = True
                    break
                row_scores[metric.name] = None
                print(f"  {row['strategy']}/{row['id']}/{metric.name} failed: {e}")

        if quota_hit:
            break  # do NOT append the partially-scored row — retry it cleanly next time

        append_jsonl(scores_path, row_scores)
        print(f"scored {row['strategy']}/{row['id']} [OK]")
        await asyncio.sleep(pacing_delay)

    print("\nDone (or stopped cleanly on quota).")


# Jupyter's native top-level await — this is the whole fix
await run_scoring_loop_async(results, metrics, SCORE_ROWS_PATH)

2 rows fully scored so far

scored naive/b1_ntk_pinn_03 [OK]
scored naive/b1_ntk_pinn_04 [OK]
  naive/b1_xpinn_01/faithfulness failed: The LLM generation was not completed. Please increase the max_tokens and try again.
scored naive/b1_xpinn_01 [OK]
scored naive/b1_xpinn_02 [OK]
  naive/b1_gns_01/faithfulness failed: The LLM generation was not completed. Please increase the max_tokens and try again.

DAILY QUOTA EXHAUSTED at naive/b1_gns_01/answer_relevancy. Resume later.


Done (or stopped cleanly on quota).


In [7]:
# Dedup and building comparison tables
import pandas as pd

rows = [json.loads(line) for line in open(SCORE_ROWS_PATH)]
scores_df = pd.DataFrame(rows).drop_duplicates(subset=["strategy", "id"], keep="last")
print(f"{len(scores_df)} unique scored rows")

metric_cols = METRIC_NAMES
print("\n=== By strategy ===")
print(scores_df.groupby("strategy")[metric_cols].mean().round(3))

print("\n=== By strategy × question type ===")
print(scores_df.groupby(["strategy", "question_type"])[metric_cols].mean().round(3))

218 unique scored rows

=== By strategy ===
               faithfulness  answer_relevancy  context_precision  \
strategy                                                           
decomposition           NaN               NaN                NaN   
hyde                    NaN               NaN                NaN   
multi_query             NaN               NaN                NaN   
naive                 0.822             0.849              0.625   
step_back               NaN               NaN                NaN   

               context_recall  
strategy                       
decomposition             NaN  
hyde                      NaN  
multi_query               NaN  
naive                   0.778  
step_back                 NaN  

=== By strategy × question type ===
                               faithfulness  answer_relevancy  \
strategy      question_type                                     
decomposition single_document           NaN               NaN   
hyde          cross_doc